# bayesflow_hpo Quickstart

A minimal end-to-end example that shows how to run **hyperparameter optimization** (HPO) for a [BayesFlow 2.x](https://bayesflow.org) amortized inference workflow.

We will:

1. Define a simple Gaussian **simulator** (prior + likelihood).
2. Build a BayesFlow **adapter** that maps raw simulation output to the format expected by the neural network.
3. Generate a fixed **validation dataset** for simulation-based calibration (SBC) diagnostics.
4. Launch an Optuna-backed **multi-objective optimization** run that searches over network architectures and training hyperparameters.

## 0. Setup

Install the package (editable mode from the repo root) and import the two libraries we need:
- **`bayesflow`** — the core amortized Bayesian inference framework (simulators, adapters, workflows),
- **`bayesflow_hpo`** — this package, which adds HPO search spaces, objectives, and validation utilities on top.

In [1]:
%pip install --quiet --upgrade -e ..

import bayesflow as bf
import bayesflow_hpo as hpo
import numpy as np

Note: you may need to restart the kernel to use updated packages.


INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).
c:\Users\Matze\Documents\GitHub\bayesflow_projects\bayesflow_hpo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Simulator, Adapter & Validation Data

**Simulator** — We define a toy generative model with a 1-D Gaussian prior $\theta \sim \mathcal{N}(0, 1)$ and a Gaussian likelihood $x_i \mid \theta \sim \mathcal{N}(\theta, 1)$ producing 12 observations per dataset. This is deliberately simple so the notebook runs in seconds.

**Adapter** — The `bf.Adapter` tells BayesFlow how to reshape the raw simulation dictionaries into the tensor format the neural network expects:
- `.as_set(["x"])` marks observation vectors as exchangeable (order doesn't matter),
- `.rename("theta", "inference_variables")` maps the parameter to the inference target,
- `.concatenate(["x"], into="summary_variables")` stacks observations into the summary input.

**Validation data** — `hpo.generate_validation_dataset` draws a fixed batch of (parameter, data) pairs from the simulator. This dataset is reused across *every* HPO trial so that metric comparisons are fair (no noise from resampling).

In [2]:
def prior_fn():
    return {"theta": np.random.normal(0.0, 1.0, size=(1,)).astype("float32")}


def likelihood_fn(theta):
    theta_value = float(np.squeeze(theta))
    x = np.random.normal(theta_value, 1.0, size=(12, 1)).astype("float32")
    return {"x": x}


simulator = bf.simulators.make_simulator([prior_fn, likelihood_fn])
adapter = (
    bf.Adapter()
    .as_set(["x"])
    .rename("theta", "inference_variables")
    .concatenate(["x"], into="summary_variables", axis=-1)
)

validation_data = hpo.generate_validation_dataset(
    simulator=simulator,
    param_keys=["theta"],
    data_keys=["x"],
    sims_per_condition=100,
)

## 2. Run HPO

`hpo.optimize` is the main entry point. Under the hood it:

1. **Creates an Optuna study** with two objectives: *calibration error* (minimize) and *model size* (minimize) — a Pareto-style trade-off between accuracy and complexity.
2. **Samples hyperparameters** from a `CompositeSearchSpace` that covers the inference network (coupling flow architecture), the summary network (DeepSet), and training settings (learning rate, batch size, etc.).
3. **Builds, trains, and validates** a fresh `bf.BasicWorkflow` for each trial, using SBC-based metrics on the fixed validation dataset.
4. **Reports results** back to Optuna, which guides future sampling via its TPE (Tree-structured Parzen Estimator) sampler.

Key arguments in this example:
| Argument | Value | Why |
|---|---|---|
| `n_trials` | 50 | Number of HPO configurations to try |
| `epochs=0` | auto | Let the search space decide epoch count (early stopping) |
| `batches_per_epoch` | 50 | Short training batches — keeps the demo fast |
| `show_progress_bar` | `False` | Avoids noisy output in notebooks |

After optimization, `study.best_trials` returns the Pareto-optimal trial(s).

In [ ]:
study = hpo.optimize(
    simulator=simulator,
    adapter=adapter,
    param_keys=["theta"],
    data_keys=["x"],
    validation_data=validation_data,
    n_trials=10,
    epochs=10,
    batches_per_epoch=20,
    max_param_count=100_000,
    show_progress_bar=False,
)

print(f"Trials: {len(study.trials)}")
print(f"Best values: {study.best_trials[0].values}")

c:\Users\Matze\Documents\GitHub\bayesflow_projects\bayesflow_hpo\.venv\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  optuna_warn(
c:\Users\Matze\Documents\GitHub\bayesflow_projects\bayesflow_hpo\.venv\Lib\site-packages\optuna\_experimental.py:33: ExperimentalWarning: Argument ``constraints_func`` is an experimental feature. The interface can change in the future.
  optuna_warn(
[I 2026-03-07 03:01:25,258] A new study created in RDB with name: bayesflow_hpo
INFO:bayesflow_hpo.optimization.objective:Trial #0 rejected: estimated 319162 params > budget 100000
[I 2026-03-07 03:01:25,379] Trial 0 finished with values: [1.0, 1.01] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 192, 'fm_subnet_depth': 3, 'fm_dropout': 0.031203728088487304, 'summary_network_type': 'deep_set', 'ds_summary_dim': 56, 'ds_depth': 3, 'ds_width': 192, 'ds_dropout': 0.006

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 159ms/step - loss: 1.3433 - moving_avg_loss: 1.3433
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 163ms/step - loss: 0.9040 - moving_avg_loss: 1.1236
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 211ms/step - loss: 0.7998 - moving_avg_loss: 1.0157
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 162ms/step - loss: 0.7554 - moving_avg_loss: 0.9506
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 175ms/step - loss: 0.6059 - moving_avg_loss: 0.8817
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 161ms/step - loss: 0.5728 - moving_avg_loss: 0.8302
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 171ms/step - loss: 0.5518 - moving_avg_loss: 0.7904
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 160ms/step - loss: 0.5331 - moving_avg_loss: 0.6747
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 158ms/step - loss: 0.4682 - moving_avg_loss: 0.6124
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 174ms/step - loss: 0.4843 - moving_avg_loss: 0.5674


INFO:bayesflow:Training completed in 34.13 seconds.
Sampling: 100%|██████████| 1/1 [00:00<00:00,  5.81batch/s]
INFO:bayesflow_hpo.optimization.objective:Trial #1 done (34s) | cal_error: 0.0379 | params: 151.8K | nrmse: 0.0871
[I 2026-03-07 03:02:00,680] Trial 1 finished with values: [0.03789473684210525, 1.0] and parameters: {'inference_network_type': 'coupling_flow', 'cf_depth': 3, 'cf_subnet_width': 64, 'cf_subnet_depth': 1, 'cf_dropout': 0.15742692948967135, 'summary_network_type': 'deep_set', 'ds_summary_dim': 41, 'ds_depth': 1, 'ds_width': 96, 'ds_dropout': 0.1099085529881075, 'initial_lr': 0.0005954553793888993}.
INFO:bayesflow_hpo.optimization.study:Progress: 1/10 trained | 1 rejected | 2 total | best: 0.0379
INFO:bayesflow_hpo.optimization.objective:Trial #2 rejected: estimated 479023 params > budget 100000
[I 2026-03-07 03:02:00,952] Trial 2 finished with values: [1.0, 1.01] and parameters: {'inference_network_type': 'coupling_flow', 'cf_depth': 5, 'cf_subnet_width': 160, 'cf_

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 219ms/step - loss: 1.4922 - moving_avg_loss: 1.4922
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 219ms/step - loss: 1.0725 - moving_avg_loss: 1.2824
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 234ms/step - loss: 0.9251 - moving_avg_loss: 1.1633
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 243ms/step - loss: 0.8281 - moving_avg_loss: 1.0795
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 278ms/step - loss: 0.6881 - moving_avg_loss: 1.0012
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 270ms/step - loss: 0.6914 - moving_avg_loss: 0.9496
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - loss: 0.5883 - moving_avg_loss: 0.8980
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 215ms/step - loss: 0.5834 - moving_avg_loss: 0.7682
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 238ms/step - loss: 0.5222 - moving_avg_loss: 0.6895
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 222ms/step - loss: 0.5322 - moving_avg_loss: 0.6334


INFO:bayesflow:Training completed in 47.58 seconds.
Sampling: 100%|██████████| 1/1 [00:00<00:00,  1.58batch/s]
INFO:bayesflow_hpo.optimization.objective:Trial #7 done (48s) | cal_error: 0.0466 | params: 117.4K | nrmse: 0.0895
[I 2026-03-07 03:02:51,294] Trial 7 finished with values: [0.0465789473684211, 1.0] and parameters: {'inference_network_type': 'coupling_flow', 'cf_depth': 5, 'cf_subnet_width': 256, 'cf_subnet_depth': 1, 'cf_dropout': 0.12311487691068891, 'summary_network_type': 'deep_set', 'ds_summary_dim': 8, 'ds_depth': 2, 'ds_width': 64, 'ds_dropout': 0.2789092957027719, 'initial_lr': 0.0023603276001278087}.
INFO:bayesflow_hpo.optimization.study:Progress: 2/10 trained | 6 rejected | 8 total | best: 0.0379
INFO:bayesflow_hpo.optimization.objective:Trial #8 rejected: estimated 352002 params > budget 100000
[I 2026-03-07 03:02:51,568] Trial 8 finished with values: [1.0, 1.01] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 224, 'fm_subnet_depth': 1

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 222ms/step - loss: 2.0012 - moving_avg_loss: 2.0012
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 208ms/step - loss: 1.4853 - moving_avg_loss: 1.7432
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 220ms/step - loss: 1.2389 - moving_avg_loss: 1.5751
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 208ms/step - loss: 1.0885 - moving_avg_loss: 1.4535
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 207ms/step - loss: 1.0252 - moving_avg_loss: 1.3678
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 209ms/step - loss: 0.9155 - moving_avg_loss: 1.2924
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 218ms/step - loss: 0.9303 - moving_avg_loss: 1.2407
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 204ms/step - loss: 0.8553 - moving_avg_loss: 1.0770
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 245ms/step - loss: 0.8355 - moving_avg_loss: 0.9842
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 289ms/step - loss: 0.8729 - moving_avg_loss: 0.9319


INFO:bayesflow:Training completed in 44.84 seconds.
Sampling: 100%|██████████| 1/1 [00:14<00:00, 14.35s/batch]
INFO:bayesflow_hpo.optimization.objective:Trial #14 done (45s) | cal_error: 0.1147 | params: 134.7K | nrmse: 0.0682
[I 2026-03-07 03:03:53,155] Trial 14 finished with values: [0.11473684210526314, 1.0] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 32, 'fm_subnet_depth': 1, 'fm_dropout': 0.10974675787331722, 'summary_network_type': 'deep_set', 'ds_summary_dim': 17, 'ds_depth': 3, 'ds_width': 64, 'ds_dropout': 0.09761990944778032, 'initial_lr': 0.0018546693963466007}.
INFO:bayesflow_hpo.optimization.objective:Trial #15 rejected: estimated 213012 params > budget 100000
[I 2026-03-07 03:03:53,401] Trial 15 finished with values: [1.0, 1.01] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 192, 'fm_subnet_depth': 3, 'fm_dropout': 0.018734953565618495, 'summary_network_type': 'deep_set', 'ds_summary_dim': 18, 'ds_depth': 

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 212ms/step - loss: 3.2912 - moving_avg_loss: 3.2912
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 222ms/step - loss: 1.6462 - moving_avg_loss: 2.4687
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 224ms/step - loss: 1.4254 - moving_avg_loss: 2.1209
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 211ms/step - loss: 1.4175 - moving_avg_loss: 1.9451
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 218ms/step - loss: 1.1703 - moving_avg_loss: 1.7901
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 222ms/step - loss: 1.0378 - moving_avg_loss: 1.6647
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 213ms/step - loss: 0.9333 - moving_avg_loss: 1.5602
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 238ms/step - loss: 0.9001 - moving_avg_loss: 1.2187
Epoch 9/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 259ms/step - loss: 0.9023 - moving_avg_loss: 1.1124
Epoch 10/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 493ms/step - loss: 0.7937 - moving_avg_loss: 1.0221


INFO:bayesflow:Training completed in 50.52 seconds.
Sampling: 100%|██████████| 1/1 [02:02<00:00, 122.30s/batch]
INFO:bayesflow_hpo.optimization.objective:Trial #22 done (51s) | cal_error: 0.3329 | params: 260.7K | nrmse: 0.1655
[I 2026-03-07 03:06:49,350] Trial 22 finished with values: [0.3328947368421053, 1.0] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 192, 'fm_subnet_depth': 3, 'fm_dropout': 0.12234414924687045, 'summary_network_type': 'deep_set', 'ds_summary_dim': 25, 'ds_depth': 4, 'ds_width': 32, 'ds_dropout': 0.03482179215207487, 'initial_lr': 0.00011971735384649119}.
INFO:bayesflow_hpo.optimization.objective:Trial #23 rejected: estimated 108624 params > budget 100000
[I 2026-03-07 03:06:49,646] Trial 23 finished with values: [1.0, 1.01] and parameters: {'inference_network_type': 'flow_matching', 'fm_subnet_width': 192, 'fm_subnet_depth': 2, 'fm_dropout': 0.019566832130200298, 'summary_network_type': 'deep_set', 'ds_summary_dim': 14, 'ds_depth'

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 191ms/step - loss: 3.7652 - moving_avg_loss: 3.7652
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 183ms/step - loss: 2.0908 - moving_avg_loss: 2.9280
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 193ms/step - loss: 1.9099 - moving_avg_loss: 2.5886
Epoch 4/10
 8/20 ━━━━━━━━━━━━━━━━━━━━ 2s 170ms/step - loss: 1.6934

## 3. Inspect Results

### 3.1 Pareto Front

The optimization minimizes *two* objectives simultaneously:
- **Objective 0** — calibration error (lower = better-calibrated posterior approximation),
- **Objective 1** — normalized parameter count (lower = smaller model).

These objectives trade off against each other: a very large network may achieve low calibration error but is expensive to run. The **Pareto front** shows the set of trials that are not dominated — no other trial is simultaneously smaller *and* better calibrated.

### 3.2 Hyperparameter Importance

After all trials have run, Optuna's **fANOVA-based importance estimator** assigns each hyperparameter a score reflecting how much its variation explains the variance in objective 0 (calibration error). High-importance hyperparameters are worth tuning carefully in follow-up studies.

### 3.3 Optimization History

The convergence plot shows how the best objective value evolves over successive trials. A healthy HPO run should trend downward — if it plateaus early, more trials won't help and you should widen the search space instead.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3.1 Pareto front: calibration error vs parameter count
hpo.plot_pareto_front(study, ax=axes[0])

# 3.2 Hyperparameter importance (fANOVA)
hpo.plot_param_importance(study, ax=axes[1])

# 3.3 Optimization history: best calibration error over trials
trained = [
    t for t in study.trials
    if t.values is not None and "rejected_reason" not in t.user_attrs
]
if trained:
    trained.sort(key=lambda t: t.number)
    trial_nums = [t.number for t in trained]
    cal_errors = [t.values[0] for t in trained]
    best_so_far = [min(cal_errors[: i + 1]) for i in range(len(cal_errors))]
    axes[2].scatter(trial_nums, cal_errors, alpha=0.4, label="Trial")
    axes[2].step(trial_nums, best_so_far, where="post", color="red", label="Best so far")
    axes[2].set_xlabel("Trial number")
    axes[2].set_ylabel("Calibration error")
    axes[2].set_title("Optimization history")
    axes[2].legend()

plt.tight_layout()
plt.show()

### 3.4 Tabular Summary

`trials_to_dataframe` converts completed trials into a `pandas.DataFrame` with one row per trial and columns for every hyperparameter, both objective values, and validation metrics (NRMSE, correlation, coverage, etc.). Budget-rejected trials are excluded by default.

The table is sorted by calibration error (the first objective) so the best architectures appear at the top.

In [ ]:
df = hpo.trials_to_dataframe(study)
print(f"Completed trials: {len(df)}")

# Show the most informative columns, sorted by calibration error
key_cols = ["trial_number", "calibration_error", "param_count", "nrmse", "correlation",
            "coverage_90", "coverage_95", "training_time_s"]
display_cols = [c for c in key_cols if c in df.columns]
df.sort_values("calibration_error")[display_cols].head(10)

### 3.5 Study Summary

`summarize_study` prints a concise overview: trial counts (trained, rejected, pruned, failed), the Pareto front, the best trial by calibration error, a top-k leaderboard, and the winning hyperparameters — everything you need to decide which configuration to use for downstream inference.

In [ ]:
hpo.summarize_study(study, top_k=5)